# Configure Smart Banking Fabric Data Agent

This notebook configures the Microsoft Fabric Data Agent for Smart Banking Risk and Fraud Prevention analytics using the `fabric-data-agent-sdk` library.

The notebook performs the following tasks:
1. **Install and Import Required Libraries** - Set up the programmatically managed SDK and dependencies
2. **Variable Initialization and AI Instructions** - Configure banking data agent settings and define master prompts
3. **Initialize Data Agent Client** - Create a connection to the Fabric Data Agent service
4. **Connect to Existing Data Agent** - Establish a connection to your pre-existing data agent instance
5. **Configure KQL Database as Data Source** - Register the BankingRiskDB KQL database and select the `real_time_transactions` table
6. **Configure Data Agent with AI Instructions and Few-shot Examples** - Apply custom banking prompts, clean existing few-shots, and upload new query templates
7. **Publish Data Agent Configuration** - Publish the changes to make the banking AI assistant ready for natural language auditing!


## Step 1: Install and Import Required Libraries

In [ ]:
# Install the fabric data agent SDK programmatically
# NOTE: In production Fabric scheduled jobs, configure this library dependency via Environment Settings
# %pip install fabric-data-agent-sdk==0.1.16a0

from uuid import UUID
from fabric.dataagent.client import FabricDataAgentManagement

print("✅ Fabric Data Agent Management SDK imported successfully!")


## Step 2: Variable Initialization and AI Instructions

Configure the real-time intelligence target credentials and define master prompts tailored to banking fraud vectors:

In [ ]:
data_agent_id = "ab287cf1-962f-4ef6-9969-a8461b5982f9"  # Replace with your Fabric Data Agent GUID if needed
kql_database_id = "a8ff77b9-3fad-449f-9f34-8eae5601c6b8"
kql_database_workspace_id = "769a23ca-8962-43fe-965d-ac1537ab0028"

print("📋 Environment Variables Loaded:")
print(f"   Workspace ID: {kql_database_workspace_id}")
print(f"   KQL DB ID:    {kql_database_id}")
print(f"   Data Agent ID: {data_agent_id}")

agent_instructions = '# Smart Banking & Fraud Prevention Data Agent - Master Prompt\n\n## Objective\n\nYou are a specialized Smart Banking Risk & Fraud Prevention analytics data agent designed to help fraud managers, compliance officers, and risk analysts query transactional records. Your primary goal is to translate natural language business questions into efficient, optimized KQL queries that isolate financial fraud, detect cross-border threats, analyze channel exposure vulnerabilities, and evaluate AI risk model performance.\n\nYour goal is to empower risk operators with real-time, data-driven security insights to safeguard customer funds, verify compliance, and perform triage triage investigations with high query execution efficiency.\n\n## Background and Solution Scope\n\nThe transactional telemetries are streamed continuously through Fabric Eventstreams into our Real-Time Intelligence Eventhouse. The system computes risk ratings for every transaction via an AI Random Forest ensemble. High-risk actions (scoring above 80%) trigger alerts and automatic freezes in accordance with RBI cyber policies.\n\nPlease adhere strictly to these operational guidelines when interacting with users:\n\n- Refrain from supplying generic financial forecasts or macroeconomic advice.\n- Do not produce graphical reports or visual chart documents directly. If requested, politely explain that you provide raw analytical datasets and that visualization is managed by the central dashboard console.\n- In responses detailing table schemas or column definitions, always exclude operational GUID fields (such as system metadata keys) unless specifically requested.\n- If a user asks general knowledge questions unrelated to banking transactions or credit risk, decline politely and restrict focus to banking and fraud analytics.\n\n## Starter Prompts\n\nSuggest the following questions for users to begin their risk assessment:\n\n- Show me a high-level overview of our transaction ingestion telemetry.\n- What are the detailed fraud statistics and vulnerability rates by payment channel?\n- Identify all cross-border transactions originating outside of India with a risk score above 80%.\n- Detect suspicious midnight transactions occurring via Internet Banking with high risk.\n- Which merchants have the highest volume of high-risk transactions?\n\n## Data Architecture & Schema\n\n**Primary Data Table:** `real_time_transactions` (ingested in real-time)\n\nKey column schemas in `real_time_transactions`:\n- `transaction_id` (string) - Unique identifier for each credit/debit transaction\n- `customer_id` / `account_number` / `customer_name` - Demographics and account links\n- `timestamp` (datetime) - ISO-8601 UTC event ingest timestamp\n- `amount` (real) - The transaction value in Indian Rupees (Rs.)\n- `merchant` / `merchant_category` - The target entity and segment (e.g., Electronics, Crypto, Luxury Shopping, ATM Cash Out)\n- `payment_channel` (string) - Channel used: UPI, POS, ATM, Mobile Banking, Internet Banking\n- `transaction_type` (string) - Typically "Debit" per the outgoing transaction scheme\n- `country` / `city` / `state` - Location telemetry (e.g., foreign countries like Russia, Ukraine, Nigeria, UAE suggest impossible travel anomaly if home location is domestic)\n- `device_type` (string) - Device profile (e.g., Unknown Linux Terminal or Suspicious Mobile Device Model)\n- `ip_address` (string) - Network footprint (e.g., Tor node IPs like 185.220.101.44)\n- `risk_score` (int) - The calculated risk percentage (0 to 100)\n- `is_fraud` (int) - Flag indicating fraud status (1 = score > 80, 0 = score <= 80)\n\n## Critical KQL Generation Guidelines\n\n### ✅ ALWAYS DO:\n1. **Time Boundaries First:** Always restrict the time window (e.g., `| where timestamp >= ago(24h)`) if specified or to optimize query scope.\n2. **Handle Timestamps Explicitly:** Cast string timestamps using `todatetime(timestamp)` before applying date/time calculations.\n3. **Use Simple Aggregations:** Construct flat queries using `summarize` grouped by the selective columns (e.g., `by payment_channel`).\n4. **Cast Values when Needed:** Ensure amount is cast using `tofloat(amount)` or `todouble(amount)` for accurate aggregates.\n5. **Optimize Joins:** Perform joins carefully; join on unique transaction keys only.\n\n### ❌ NEVER DO:\n1. **Cartesian Product Joins:** Avoid joining large transaction subsets without strict filters.\n2. **Circular calculated columns:** Do not reference a column initialized in the same `extend` block.\n3. **Complex Nested Subqueries:** Avoid writing deeply nested operations; keep KQL straightforward and highly readable.\n4. **Non-operational aggregates:** Don\'t aggregate non-numeric fields like string categories.\n\n## Proven KQL Query Patterns\n\n**High-Level Telemetry Overview:**\n```kql\nreal_time_transactions\n| summarize \n    TotalTransactions = count(),\n    DateFrom = min(todatetime(timestamp)),\n    DateTo = max(todatetime(timestamp)),\n    UniqueCustomers = dcount(customer_id),\n    AvgAmount = round(avg(amount), 2),\n    AvgRisk = round(avg(risk_score), 1),\n    FraudCount = countif(risk_score > 80)\n```\n\n**Vulnerability Risk per Channel:**\n```kql\nreal_time_transactions\n| summarize \n    TotalTransactions = count(),\n    TotalFraud = countif(risk_score > 80),\n    MaxRisk = max(risk_score),\n    AvgRisk = round(avg(risk_score), 2),\n    TotalAmount = round(sum(amount), 2)\nby payment_channel\n| extend FraudPercentage = round(TotalFraud * 100.0 / TotalTransactions, 2)\n| order by FraudPercentage desc\n```\n\n**Cross-Border Threat Vectors:**\n```kql\nreal_time_transactions\n| where country != "India" and risk_score > 80\n| project timestamp, transaction_id, customer_name, country, amount, risk_score, payment_channel, device_type\n| order by risk_score desc\n```\n\n**Suspicious Midnight Taking Anomalies:**\n```kql\nreal_time_transactions\n| extend Hour = hourofday(todatetime(timestamp))\n| where Hour in (1, 2, 3, 4) and payment_channel == "Internet Banking" and risk_score > 80\n| project timestamp, transaction_id, customer_name, amount, risk_score, ip_address, device_type\n| order by risk_score desc\n```\n'
data_source_instructions = '# Data Source Instructions - Banking Operations\n\n## Overview\nThis Fabric Data Agent accesses the Contoso Smart Banking Risk & Fraud Telemetry database inside a Microsoft Fabric Eventhouse.\n\n## Business Context\nThe data comprises continuous transactional streams ingested via Azure Event Hubs / Fabric Eventstreams. The primary ledger tracks outbound debits, customer devices, and risk scores computed in real-time.\n\n### Database Tables and Relationships\n- **`real_time_transactions`** - Transaction telemetry records containing security scores, transaction sizes, geolocation coordinates, and network IP addresses.\n\n### Primary Query Goal\nAssist risk investigators to evaluate and isolate fraudulent operations by channel, identify cross-border AML anomalies, flag routing through malicious Tor nodes, and verify operational thresholds.\n'
data_source_description = "# Data Source Descriptions - Smart Banking Risk\n\n## Overview\nReal-time transactional audit table for Contoso Smart Banking's retail credit card, net-banking, ATM, and UPI networks in India.\n\n## Data Tables\n### Table `real_time_transactions`\nContinuous event log of customer transaction telemetries. Records include transaction IDs, customer accounts, amounts, merchant details, devices, geographic coordinates, IP addresses, AI model risk scores, and compliance tags.\n"
fewshots_examples = {'Show me a high-level overview of our transaction ingestion telemetry.': 'real_time_transactions\r\n| summarize \r\n    TotalTransactions = count(),\r\n    DateFrom = min(todatetime(timestamp)),\r\n    DateTo = max(todatetime(timestamp)),\r\n    UniqueCustomers = dcount(customer_id),\r\n    AvgAmount = round(avg(amount), 2),\r\n    AvgRisk = round(avg(risk_score), 1),\r\n    FraudCount = countif(risk_score > 80)', 'What are the detailed fraud statistics and vulnerability rates by payment channel?': 'real_time_transactions\r\n| summarize \r\n    TotalTransactions = count(),\r\n    TotalFraud = countif(risk_score > 80),\r\n    MaxRisk = max(risk_score),\r\n    AvgRisk = round(avg(risk_score), 2),\r\n    TotalAmount = round(sum(amount), 2)\r\nby payment_channel\r\n| extend FraudPercentage = round(TotalFraud * 100.0 / TotalTransactions, 2)\r\n| order by FraudPercentage desc', 'Identify all cross-border transactions originating outside of India with a risk score above 80%.': 'real_time_transactions\r\n| where country != "India" and risk_score > 80\r\n| project timestamp, transaction_id, customer_name, country, amount, risk_score, payment_channel, device_type\r\n| order by risk_score desc', 'Detect suspicious midnight transactions occurring via Internet Banking with high risk.': 'real_time_transactions\r\n| extend Hour = hourofday(todatetime(timestamp))\r\n| where Hour in (1, 2, 3, 4) and payment_channel == "Internet Banking" and risk_score > 80\r\n| project timestamp, transaction_id, customer_name, amount, risk_score, ip_address, device_type\r\n| order by risk_score desc'}

print(f"\n📝 Prompts and instructions loaded:")
print(f"   Master Instructions:     {len(agent_instructions)} characters")
print(f"   KQL Source Instructions: {len(data_source_instructions)} characters")
print(f"   KQL Source Description:  {len(data_source_description)} characters")
print(f"   Few-shot query examples: {len(fewshots_examples)} registered")
print(f"✅ Configuration vectors prepared")


## Step 3: Initialize Data Agent Client

In [ ]:
# Initialize the SDK client for data agent management
mgmt_client = FabricDataAgentManagement(UUID(data_agent_id))
print(f"✅ Successfully initialized management client connection for Agent ID: {data_agent_id}")


## Step 4: Connect to Existing Data Agent and Verify Status

In [ ]:
print(f"🤖 Connecting and fetching current agent configuration...")
config = mgmt_client.get_configuration()
print(f"✅ Successfully connected to Fabric Data Agent!")
print(f"   Agent Status: Ready for configuration updates")


## Step 5: Configure KQL Database as Data Source

Bind the Eventhouse KQL Database and enable target tables for transactional AI auditing:

In [ ]:
print(f"🔗 Attaching KQL Database as Data Source...")

datasource = mgmt_client.add_datasource(
    workspace_id_or_name=UUID(kql_database_workspace_id),
    artifact_name_or_id=UUID(kql_database_id),
    type="kqldatabase"
)

print(f"✅ Successfully added KQL Database data source!")
print(f"   Datasource ID: {datasource._id}")

# Select the target real-time transaction table for the agent
selected_tables = ["real_time_transactions"]
print(f"\n📋 Enabling transaction tables for AI access...")
for table in selected_tables:
    datasource.select(table)
    print(f"   ✓ Table registered: {table}")

print(f"✅ Table configuration completed successfully!")


## Step 6: Configure Data Agent with Banking Prompts and Few-shot Examples

Apply the master cyber-fraud prompts, clear historical cached few-shots, and upload the new smart banking templates:

In [ ]:
print(f"🤖 Updating Data Agent configurations with master prompts...")
mgmt_client.update_configuration(instructions=agent_instructions)
print(f"   ✓ Master instructions applied!")

print(f"\n🔗 Updating KQL datasource details and annotations...")
datasource.update_configuration(
    instructions=data_source_instructions,
    user_description=data_source_description
)
print(f"   ✓ Datasource context instructions applied!")

# Clean up prior few-shot history
print(f"\n🔍 Auditing historical few-shot database...")
existing_fewshots = datasource.get_fewshots()
print(f"   Found {len(existing_fewshots)} old few-shot examples")

if len(existing_fewshots) > 0:
    print(f"🗑️ Purging cached few-shot tables...")
    for i, row in existing_fewshots.iterrows():
        fs_id = row['Id']
        question = row['Question'][:50] + "..." if len(row['Question']) > 50 else row['Question']
        print(f"   Removing: {question}")
        datasource.remove_fewshot(fs_id)
    print(f"   ✓ Purged old examples")
else:
    print(f"   ✓ Cache clear")

# Upload new structured KQL templates
print(f"\n📚 Injecting Smart Banking few-shot KQL templates...")
for idx, (question, query) in enumerate(fewshots_examples.items(), 1):
    truncated_q = question[:60] + "..." if len(question) > 60 else question
    print(f"   {idx}. Uploading: {truncated_q}")
    datasource.add_fewshots({question: query})

print(f"✅ Successfully registered all {len(fewshots_examples)} few-shot pairs!")

# Final structural audit
final_fewshots_df = datasource.get_fewshots()
config_audit = mgmt_client.get_configuration()

print(f"\n📊 Configuration Audit Verification:")
print(f"   Agent Instructions Applied: {'✓' if config_audit.instructions else '✗'}")
print(f"   Active Few-shot templates:  {len(final_fewshots_df)}")


## Step 7: Publish Data Agent Configuration

Publish all configurations to deploy the agent, making it active for natural language telemetry query resolution in Fabric:

In [ ]:
print(f"📤 Publishing and deploying Banking Risk & Fraud Intelligence Agent...")
mgmt_client.publish()
print(f"\n🎉 SUCCESS: Data Agent published and deployed!")
print(f"   Agent ID:       {data_agent_id}")
print(f"   Enabled Table:  {', '.join(selected_tables)}")
print(f"   Status:         Active & Ready for Business Queries")
